# 01 — Environment Setup & Data Loading

**Pre-condition:** Run `prepare_data.py` locally before this notebook.
It converts BDD100K into filtered YOLO/COCO splits and writes them to
`data_prepared/`. Upload that folder to Google Drive once.

This notebook:
1. Installs dependencies
2. Verifies the GPU environment
3. Mounts Drive and copies the pre-converted splits to `/content/`
4. Validates split image/label counts

In [ ]:
!pip install -q ultralytics rfdetr fvcore nvidia-ml-py python-dotenv
!pip install -q /content/drive/MyDrive/FON/master_rad/bdd100k_data/thesis_src-0.1.0-py3-none-any.whl

In [ ]:
import torch
from dotenv import load_dotenv
load_dotenv('/content/drive/MyDrive/FON/master_rad/bdd100k_data/.env')

print('Python:', __import__('sys').version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Mount Google Drive — the pre-converted data_prepared/ folder must be here
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = '/content/drive/MyDrive/FON/master_rad/bdd100k_data'

In [ ]:
from pathlib import Path
import os, zipfile

DRIVE_ROOT   = Path('/content/drive/MyDrive/FON/master_rad/bdd100k_data')
DATA_ROOT    = Path('/content/data_prepared')
YOLO_DATA_DIR = DATA_ROOT / 'yolo'
COCO_DATA_DIR = DATA_ROOT / 'coco'
CONFIGS_DIR   = DATA_ROOT / 'configs'

ZIP_SRC = DRIVE_ROOT / 'bdd100k_data.zip'

if not YOLO_DATA_DIR.exists():
    print('Extracting...')
    with zipfile.ZipFile(ZIP_SRC) as z:
        z.extractall('/content/')
    print('Done.')
else:
    print('Already extracted — skipping.')

In [ ]:
# Validate split contents
splits = [
    'clear_day/train', 'clear_day/val',
    'rainy_day', 'snowy_day', 'night_clear',
    'overcast_day', 'partly_cloudy_day', 'dawn_dusk_clear',
]
expected = {
    'clear_day/train':    12400,
    'clear_day/val':       3500,
    'rainy_day':            396,
    'snowy_day':            422,
    'night_clear':         3274,
    'overcast_day':        1039,
    'partly_cloudy_day':    638,
    'dawn_dusk_clear':      307,
}

print(f'{"Split":<25} {"Images":>8} {"Labels":>8} {"Expected":>10}')
print('-' * 55)
for split in splits:
    img_dir = YOLO_DATA_DIR / split / 'images'
    lbl_dir = YOLO_DATA_DIR / split / 'labels'
    n_img = len(list(img_dir.glob('*.jpg'))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    exp   = expected.get(split, '?')
    ok    = '✓' if abs(n_img - exp) < 30 else '✗'
    print(f'{split:<25} {n_img:>8} {n_lbl:>8} {exp:>10}  {ok}')